# 📖 Notebook 3: Ledger & Double-Entry Bookkeeping

In a payment system, every dollar must be accounted for. If your numbers don't add up, you have a bug — or fraud.  
The solution used by every financial system since the 15th century is **double-entry bookkeeping**.

## The Core Rule

Every time money moves, we record **two** entries:
- A **debit** (money coming from somewhere)
- A **credit** (money going to somewhere)

The total of all debits must **always** equal the total of all credits. If they don't, something is wrong.

## Learning Objectives

By the end of this notebook you'll understand:
- Why double-entry bookkeeping exists and how it catches errors
- How to model ledger entries in a database
- How to verify the books balance (debits == credits)
- How to query account balances and transaction history

## 🛠️ Setup

```bash
cd system-designs/payment-system
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import uuid

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "payment_demo", "user": "demo", "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

conn = get_db()
print("✅ Postgres connected")
conn.close()

✅ Postgres connected


---
## 1. Why Double-Entry? A Simple Analogy

Think of it like moving a ball between two boxes:

- **Box A** (the customer's wallet) loses the ball → that's a **debit** on their account.
- **Box B** (the merchant's account) gains the ball → that's a **credit** on their account.

If you ever count all the balls and find the total has changed, something went wrong.  
Double-entry bookkeeping is just a formal way of making sure **no balls disappear or appear from nowhere**.

### Our Accounts

For a simplified payment system, we use two main accounts:

| Account | What It Represents |
|---------|-------------------|
| `customer_receivable` | Money we expect to collect from the customer's card |
| `merchant_payable` | Money we owe to the merchant |

When a charge succeeds:
- **Debit** `customer_receivable` (we received money from the customer)
- **Credit** `merchant_payable` (we owe that money to the merchant)

---
## 2. Bad Practice First: Single-Entry Bookkeeping

Before we admire double-entry, let's see what happens **without** it.

Naive systems just record `merchant X earned $Y` in a single `earnings` table. One row, one number. Simple, right? The problem:

- If the row is wrong (typo, bug, malicious edit), there is **nothing to compare it to**.
- You cannot tell where the money came from or where it went -- there is no `other side`.
- You cannot detect partial writes: if the app crashes mid-save, the single number just ends up wrong with no alarm.

Let's build a tiny single-entry version and see why it breaks.


In [2]:
# BAD: single-entry bookkeeping -- one row per payment, no counterpart.
# We store everything in a throwaway temp table so we don't pollute our real schema.
conn = get_db()
cur = conn.cursor()
cur.execute("CREATE TEMP TABLE IF NOT EXISTS earnings_single_entry (merchant_id TEXT, amount_cents INTEGER)")
cur.execute("TRUNCATE earnings_single_entry")

# Record 3 legitimate payments...
for merchant, amount in [("merch_001", 5000), ("merch_002", 2500), ("merch_001", 1000)]:
    cur.execute("INSERT INTO earnings_single_entry VALUES (%s, %s)", (merchant, amount))

# ...and now simulate a bug: one row gets written with the wrong amount (off-by-one-zero).
cur.execute("INSERT INTO earnings_single_entry VALUES ('merch_002', 99999)")  # should have been 9999

cur.execute("SELECT merchant_id, SUM(amount_cents) FROM earnings_single_entry GROUP BY merchant_id ORDER BY merchant_id")
print("Single-entry 'earnings' table:")
for row in cur.fetchall():
    print(f"  {row[0]}: ${row[1]/100:.2f}")

print()
print("Problem: nothing in this schema can tell us row #4 is wrong.")
print("There is no counterpart entry to compare against. The books 'balance'")
print("against nothing, because there is no other side.")
cur.close(); conn.close()


Single-entry 'earnings' table:
  merch_001: $60.00
  merch_002: $1024.99

Problem: nothing in this schema can tell us row #4 is wrong.
There is no counterpart entry to compare against. The books 'balance'
against nothing, because there is no other side.


---
## 3. Examining the Existing Ledger

Our `init.sql` already seeded some ledger entries. Let's look at them.

In [3]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Show all ledger entries
cur.execute("""
    SELECT le.transaction_id, le.account_name, le.entry_type, 
           le.amount_cents, le.description
    FROM ledger_entries le
    ORDER BY le.transaction_id, le.entry_type
""")

rows = cur.fetchall()
print(f"Total ledger entries: {len(rows)}\n")
print(f"{'Transaction':<15} {'Account':<25} {'Type':<8} {'Amount':>10}  Description")
print("-" * 90)
for r in rows:
    amt = f"${r['amount_cents']/100:.2f}"
    print(f"{r['transaction_id']:<15} {r['account_name']:<25} {r['entry_type']:<8} {amt:>10}  {r['description']}")

cur.close(); conn.close()

Total ledger entries: 12

Transaction     Account                   Type         Amount  Description
------------------------------------------------------------------------------------------
txn_001         merchant_payable          credit       $49.99  Charge for pi_001
txn_001         customer_receivable       debit        $49.99  Charge for pi_001
txn_002         merchant_payable          credit       $12.99  Charge for pi_002
txn_002         customer_receivable       debit        $12.99  Charge for pi_002
txn_003         merchant_payable          credit      $249.99  Charge for pi_003
txn_003         customer_receivable       debit       $249.99  Charge for pi_003
txn_006         merchant_payable          credit       $34.99  Charge for pi_007
txn_006         customer_receivable       debit        $34.99  Charge for pi_007
txn_007         merchant_payable          credit       $24.99  Charge for pi_008
txn_007         customer_receivable       debit        $24.99  Charge for pi_00

Notice the pattern: **every transaction has exactly two entries** — one debit and one credit for the same amount.

---
## 4. The Balance Check: Do the Books Balance?

The fundamental accounting equation:  
**Total Debits = Total Credits**

If this ever fails, we have a bug. Let's verify.

In [4]:
def check_books_balance():
    """Verify that total debits equal total credits across all ledger entries."""
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""
        SELECT 
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits
        FROM ledger_entries
    """)
    row = cur.fetchone()
    total_debits = row[0] or 0
    total_credits = row[1] or 0

    print(f"Total Debits  : ${total_debits/100:.2f}")
    print(f"Total Credits : ${total_credits/100:.2f}")
    print(f"Difference    : ${(total_debits - total_credits)/100:.2f}")

    if total_debits == total_credits:
        print("\n✅ Books balance! Every dollar is accounted for.")
    else:
        print("\n🚨 BOOKS DO NOT BALANCE! Something is wrong!")

    cur.close(); conn.close()
    return total_debits == total_credits

check_books_balance()

Total Debits  : $522.95
Total Credits : $522.95
Difference    : $0.00

✅ Books balance! Every dollar is accounted for.


True

---
## 5. Recording a New Charge with Ledger Entries

Let's build a function that processes a charge and records both ledger entries in a single database transaction.  
Using a database transaction (`BEGIN ... COMMIT`) ensures that either **both** entries are written, or **neither** is. We never end up with a debit without its matching credit.

In [5]:
def record_charge_with_ledger(merchant_id, amount_cents, description):
    """
    Create a PaymentIntent, a Transaction, and the matching ledger entries.
    Everything happens in a single database transaction for atomicity.
    """
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        # Create PaymentIntent (succeeded immediately for this demo)
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
            VALUES (%s, %s, %s, 'usd', %s, 'succeeded')
        """, (pi_id, merchant_id, amount_cents, description))

        # Create Transaction
        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'charge', %s, 'usd', 'succeeded', '4242', 'visa')
        """, (txn_id, pi_id, amount_cents))

        # Double-entry ledger: DEBIT customer_receivable, CREDIT merchant_payable
        cur.execute("""
            INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
            VALUES
                (%s, 'customer_receivable', 'debit',  %s, 'usd', %s),
                (%s, 'merchant_payable',    'credit', %s, 'usd', %s)
        """, (txn_id, amount_cents, f"Charge for {pi_id}",
              txn_id, amount_cents, f"Charge for {pi_id}"))

        conn.commit()
        print(f"✅ Charge recorded:")
        print(f"   PaymentIntent : {pi_id}")
        print(f"   Transaction   : {txn_id}")
        print(f"   Amount        : ${amount_cents/100:.2f}")
        print(f"   Ledger        : DEBIT customer_receivable ${amount_cents/100:.2f}")
        print(f"                   CREDIT merchant_payable   ${amount_cents/100:.2f}")
        return txn_id

    except Exception as e:
        conn.rollback()
        print(f"❌ Error: {e}")
        return None

    finally:
        cur.close(); conn.close()

# Record a few charges
record_charge_with_ledger("merch_001", 5999, "Ledger demo: premium headphones")
print()
record_charge_with_ledger("merch_002", 1250, "Ledger demo: basic widget")

✅ Charge recorded:
   PaymentIntent : pi_95fe45a43b1c
   Transaction   : txn_d66b5240e90e
   Amount        : $59.99
   Ledger        : DEBIT customer_receivable $59.99
                   CREDIT merchant_payable   $59.99

✅ Charge recorded:
   PaymentIntent : pi_6cb42234c216
   Transaction   : txn_1480bd7107cd
   Amount        : $12.50
   Ledger        : DEBIT customer_receivable $12.50
                   CREDIT merchant_payable   $12.50


'txn_1480bd7107cd'

In [6]:
# Verify the books still balance after our new charges
check_books_balance()

Total Debits  : $595.44
Total Credits : $595.44
Difference    : $0.00

✅ Books balance! Every dollar is accounted for.


True

---
## 6. Account Balances

We can compute the balance of any account by summing its debits and credits.  
This is how you'd answer questions like "how much do we owe merchant X?" or "how much have we collected from customers?"

In [7]:
def show_account_balances():
    """Show the balance of each account in the ledger."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT 
            account_name,
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits,
            COUNT(*) AS entry_count
        FROM ledger_entries
        GROUP BY account_name
        ORDER BY account_name
    """)

    print(f"{'Account':<25} {'Debits':>12} {'Credits':>12} {'Net':>12}  Entries")
    print("-" * 75)
    for r in cur.fetchall():
        net = r['total_debits'] - r['total_credits']
        print(f"{r['account_name']:<25} ${r['total_debits']/100:>10.2f} ${r['total_credits']/100:>10.2f} ${net/100:>10.2f}  {r['entry_count']}")

    cur.close(); conn.close()

show_account_balances()

Account                         Debits      Credits          Net  Entries
---------------------------------------------------------------------------
customer_receivable       $    595.44 $      0.00 $    595.44  8
merchant_payable          $      0.00 $    595.44 $   -595.44  8


---
## 7. What Happens When the Books Don't Balance?

Let's intentionally break the ledger by inserting a single entry without its matching pair.  
This simulates a bug where the credit entry wasn't written (e.g., the app crashed between the two inserts).

In [8]:
# Simulate a bug: the app creates a Transaction and writes the DEBIT entry,
# then crashes BEFORE writing the matching CREDIT. Without a safety net the
# ledger ends up with an orphan debit.
import uuid
bug_pi_id  = f"pi_{uuid.uuid4().hex[:12]}"
bug_txn_id = f"txn_{uuid.uuid4().hex[:12]}"

conn = get_db()
cur = conn.cursor()
# Seed a real payment_intent + transaction so the FK is satisfied (this part
# of the code "worked"). The missing credit is what simulates the crash.
cur.execute("""
    INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
    VALUES (%s, 'merch_001', 9999, 'usd', 'BUG demo: app crashed mid-write', 'succeeded')
""", (bug_pi_id,))
cur.execute("""
    INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
    VALUES (%s, %s, 'charge', 9999, 'usd', 'succeeded', '4242', 'visa')
""", (bug_txn_id, bug_pi_id))

# Now write ONLY the debit -- the credit is "lost" to the crash
cur.execute("""
    INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
    VALUES (%s, 'customer_receivable', 'debit', 9999, 'usd', 'BUG: missing credit entry')
""", (bug_txn_id,))
conn.commit()
cur.close(); conn.close()

print(f"Inserted a debit on {bug_txn_id} with NO matching credit...")
print()
check_books_balance()


Inserted a debit on txn_13b8dfca295a with NO matching credit...



Total Debits  : $695.43
Total Credits : $595.44
Difference    : $99.99

🚨 BOOKS DO NOT BALANCE! Something is wrong!


False

In [9]:
# Find the broken entry
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Find transactions where debits != credits
cur.execute("""
    SELECT 
        transaction_id,
        SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS debits,
        SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS credits
    FROM ledger_entries
    GROUP BY transaction_id
    HAVING SUM(CASE WHEN entry_type = 'debit' THEN amount_cents ELSE 0 END)
        != SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END)
""")

broken = cur.fetchall()
print("🔍 Transactions where debits ≠ credits:")
for r in broken:
    print(f"  {r['transaction_id']}: debits=${r['debits']/100:.2f}  credits=${r['credits']/100:.2f}")

cur.close(); conn.close()

🔍 Transactions where debits ≠ credits:
  txn_13b8dfca295a: debits=$99.99  credits=$0.00


In [10]:
# Clean up the broken entry, and the transaction/intent we created for the demo.
# Order matters because of foreign keys: ledger_entries -> transactions -> payment_intents.
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM ledger_entries WHERE transaction_id = %s", (bug_txn_id,))
cur.execute("DELETE FROM transactions    WHERE id = %s", (bug_txn_id,))
cur.execute("DELETE FROM payment_intents WHERE id = %s", (bug_pi_id,))
conn.commit()
cur.close(); conn.close()

print("Cleaned up the broken entry and its parent rows.")
check_books_balance()


Cleaned up the broken entry and its parent rows.
Total Debits  : $595.44
Total Credits : $595.44
Difference    : $0.00

✅ Books balance! Every dollar is accounted for.


True

---
## 8. Per-Merchant Balances

In a real system, you'd want to see how much each merchant has earned.  
We can join the ledger with transactions and payment intents to break down balances by merchant.

In [11]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT 
        m.name AS merchant_name,
        COUNT(DISTINCT le.transaction_id) AS transaction_count,
        SUM(CASE WHEN le.entry_type = 'credit' AND le.account_name = 'merchant_payable' 
            THEN le.amount_cents ELSE 0 END) AS total_earned_cents
    FROM ledger_entries le
    JOIN transactions t ON le.transaction_id = t.id
    JOIN payment_intents pi ON t.payment_intent_id = pi.id
    JOIN merchants m ON pi.merchant_id = m.id
    GROUP BY m.name
    ORDER BY total_earned_cents DESC
""")

print(f"{'Merchant':<25} {'Transactions':>15} {'Total Earned':>15}")
print("-" * 60)
for r in cur.fetchall():
    print(f"{r['merchant_name']:<25} {r['transaction_count']:>15} ${r['total_earned_cents']/100:>13.2f}")

cur.close(); conn.close()

Merchant                     Transactions    Total Earned
------------------------------------------------------------
Widget Co                               3 $       412.49
Acme Online Store                       4 $       147.96
Book Haven                              1 $        34.99


---
## 8. Refunds: Reversing the Ledger

Real payment systems need refunds. In double-entry land, a refund does not delete the original charge -- the ledger is **append-only**. Instead, we write **two new rows** that reverse the original: DEBIT `merchant_payable`, CREDIT `customer_receivable`.

After the refund, both accounts net to zero for that transaction, and anyone auditing the books can still see the full history (charge + refund), not just the final result.


In [12]:
def record_refund(original_txn_id):
    """
    Issue a full refund by writing a reversing pair of ledger entries.
    We never modify or delete the original rows (append-only ledger).
    """
    refund_txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()
    try:
        cur.execute("""
            SELECT payment_intent_id, amount_cents, currency
            FROM transactions WHERE id = %s
        """, (original_txn_id,))
        row = cur.fetchone()
        if not row:
            print(f"Transaction {original_txn_id} not found"); return None
        pi_id, amount_cents, currency = row

        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'refund', %s, %s, 'succeeded', '4242', 'visa')
        """, (refund_txn_id, pi_id, amount_cents, currency))

        # Reversing ledger entries: same accounts, debit <-> credit swapped
        cur.execute("""
            INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
            VALUES
                (%s, 'merchant_payable',    'debit',  %s, %s, %s),
                (%s, 'customer_receivable', 'credit', %s, %s, %s)
        """, (refund_txn_id, amount_cents, currency, f"Refund of {original_txn_id}",
              refund_txn_id, amount_cents, currency, f"Refund of {original_txn_id}"))
        conn.commit()
        print("Refund recorded:")
        print(f"   Original      : {original_txn_id}  (${amount_cents/100:.2f})")
        print(f"   Refund txn    : {refund_txn_id}")
        print(f"   Ledger        : DEBIT  merchant_payable    ${amount_cents/100:.2f}")
        print(f"                   CREDIT customer_receivable ${amount_cents/100:.2f}")
        return refund_txn_id
    except Exception as e:
        conn.rollback(); print(f"Error: {e}"); return None
    finally:
        cur.close(); conn.close()

# Refund one of our seeded charges (txn_001 = $49.99)
record_refund("txn_001")
print()
check_books_balance()


Refund recorded:
   Original      : txn_001  ($49.99)
   Refund txn    : txn_9535e88691b7
   Ledger        : DEBIT  merchant_payable    $49.99
                   CREDIT customer_receivable $49.99



Total Debits  : $645.43
Total Credits : $645.43
Difference    : $0.00

✅ Books balance! Every dollar is accounted for.


True

In [13]:
# Zoom in on txn_001 + its refund: the charge and reversal together should net
# to ZERO on every account touched. That is the property auditors check.
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT le.transaction_id, t.type, le.account_name, le.entry_type, le.amount_cents
    FROM ledger_entries le
    JOIN transactions t ON t.id = le.transaction_id
    WHERE t.payment_intent_id = 'pi_001'
    ORDER BY le.transaction_id, le.entry_type
""")
for r in cur.fetchall():
    sign = "+" if r['entry_type']=='debit' else "-"
    print(f"  {r['transaction_id']:<16} {r['type']:<7} {r['account_name']:<22} {r['entry_type']:<6} {sign}${r['amount_cents']/100:.2f}")

cur.execute("""
    SELECT le.account_name,
           SUM(CASE WHEN le.entry_type='debit'  THEN le.amount_cents ELSE 0 END)
         - SUM(CASE WHEN le.entry_type='credit' THEN le.amount_cents ELSE 0 END) AS net_cents
    FROM ledger_entries le
    JOIN transactions t ON t.id = le.transaction_id
    WHERE t.payment_intent_id = 'pi_001'
    GROUP BY le.account_name
""")
print()
print("Net per account for pi_001 (should be $0.00 after refund):")
for r in cur.fetchall():
    print(f"  {r['account_name']:<22} ${r['net_cents']/100:.2f}")
cur.close(); conn.close()


  txn_001          charge  merchant_payable       credit -$49.99


  txn_001          charge  customer_receivable    debit  +$49.99
  txn_9535e88691b7 refund  customer_receivable    credit -$49.99
  txn_9535e88691b7 refund  merchant_payable       debit  +$49.99

Net per account for pi_001 (should be $0.00 after refund):
  customer_receivable    $0.00
  merchant_payable       $0.00


---
## 9. Summary

| Concept | Key Point |
|---------|----------|
| **Double-Entry** | Every money movement creates two entries: a debit and a credit |
| **Balance Check** | Total debits must always equal total credits |
| **Atomicity** | Both entries are written in a single DB transaction — all or nothing |
| **Auditability** | The ledger is append-only — you never update or delete entries |
| **Error Detection** | If debits ≠ credits for any transaction, you've found a bug |

### Why This Matters in System Design Interviews

When an interviewer asks "how do you ensure financial integrity?", the answer is:
1. **Double-entry bookkeeping** to track every dollar.
2. **Database transactions** to ensure atomic writes.
3. **Balance verification queries** to catch bugs.
4. **Append-only audit logs** for compliance.

➡️  Next notebook: **Fraud Detection Basics**